In [ ]:
# Imports
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import shap
import matplotlib.pyplot as plt

# --- Paths ---
data_path = "../data/MachineLearningRating_v3.txt"
outdir = "../reports/task4"
os.makedirs(outdir, exist_ok=True)

# --- Load data ---
df = load_data(data_path)

# --- Feature engineering ---
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)
df['Margin'] = df['TotalPremium'] - df['TotalClaims']

# Filter to rows with claims for severity prediction
df_claims = df[df['HasClaim'] == 1].copy()

# Select features for modeling
features_cat = ['Province','VehicleType','make','Model','Gender','CoverCategory','CoverType']
features_num = ['TotalPremium','CalculatedPremiumPerTerm','CustomValueEstimate','Cylinders','cubiccapacity','kilowatts','NumberOfDoors']
target = 'TotalClaims'

# Keep only columns present in data
features_cat = [f for f in features_cat if f in df_claims.columns]
features_num = [f for f in features_num if f in df_claims.columns]

# Drop rows with missing target
df_claims = df_claims.dropna(subset=[target])

# --- Train-test split ---
X = df_claims[features_cat + features_num]
y = df_claims[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Preprocessing ---
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer(transformers=[
    ('cat', cat_transformer, features_cat),
    ('num', num_transformer, features_num)
])

# --- Models ---
models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, objective='reg:squarederror')
}

# --- Train, evaluate, save results ---
results = {}
for name, model in models.items():
    print(f"Training {name}...")
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    r2 = r2_score(y_test, y_pred)
    results[name] = {'RMSE': rmse, 'R2': r2, 'model': pipe}
    print(f"{name} -> RMSE: {rmse:.2f}, R2: {r2:.3f}")

# --- SHAP Feature Importance for best model (choose lowest RMSE) ---
best_model_name = min(results, key=lambda k: results[k]['RMSE'])
best_pipe = results[best_model_name]['model']
print(f"\nBest model based on RMSE: {best_model_name}")

# Get preprocessed feature names
ohe_features = best_pipe.named_steps['preprocessor'].transformers_[0][1] \
                .named_steps['encoder'].get_feature_names_out(features_cat)
feature_names = np.concatenate([ohe_features, features_num])

# SHAP explanation
explainer = shap.Explainer(best_pipe.named_steps['regressor'], 
                           best_pipe.named_steps['preprocessor'].transform(X_train))
shap_values = explainer(best_pipe.named_steps['preprocessor'].transform(X_test))

# Plot top 10 important features
shap.summary_plot(shap_values, features=best_pipe.named_steps['preprocessor'].transform(X_test),
                  feature_names=feature_names, max_display=10, show=True)

# Save results summary
pd.DataFrame({k: {'RMSE': v['RMSE'], 'R2': v['R2']} for k,v in results.items()}).to_csv(os.path.join(outdir,'model_performance.csv'))
print("Model performance saved to:", outdir)
